# 纳米棒自组装：分析

| 文件 | 用途 |
|------|------|
| `log.lammps` | 温度 / 压力 / 边长 / 体积 / 密度 |
| `result_atoms.eq.data` + `result_atoms.lammpstrj` | 加压段轨迹 → 可视化 / $S_2$ / fresnel |

- `_helper_functions.py` — `load_lammps_universe` / thermo / `save_nglview_frame`
- `_nematic_order.py` — $S_2$；相对密度 $\rho^*=\rho/\rho_{cp}$
- `_render_by_fresnel.py` — 指定帧路径追踪渲染并保存 PNG

**相对密度：** 相对 close packing 棒数密度
$\rho^*=\rho/\rho_{cp}$，
$\rho_{cp}=2\big/\big(\sqrt{2}+(L/D)\sqrt{3}\big)$，
$\rho=N_\mathrm{rod}/V$。
LAMMPS `density`（$m=1$）为 $\rho_\mathrm{atom}$，故 $\rho=\rho_\mathrm{atom}/N_\mathrm{beads}$。

顺序：**可视化 → 热力学 → $\rho^*$–时间 / $\rho^*$–压力 → $S_2$ → 汇总 → fresnel 渲染**。


In [ ]:
%reload_ext autoreload
%autoreload 2
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = "retina"
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 200,
    "figure.facecolor": "white",
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.2,
    "font.size": 10,
})
from MDAnalysis import transformations as trans
import nglview as nv
import warnings
warnings.filterwarnings("ignore")

from _helper_functions import (
    load_lammps_universe,
    read_result_thermo,
    move_origin_to_corner,
    save_nglview_frame,
)
import _nematic_order as nem
from _render_by_fresnel import render_universe_frame, render_snapshot_series, color_props_rod_alignment

# ---- 本例参数（换体系改这里）----
TOPO = "./result_atoms.eq.data"   # 加压段起点构型
TRAJ = "./result_atoms.lammpstrj"
N_BEADS = 11
ROD_L = 5.0                       # spherocylinder 圆柱段长 L
ROD_D = 1.0                       # 直径 D (= σ)
RHO_CP = nem.close_packing_density(ROD_L, ROD_D)
print(f"rho_cp = {RHO_CP:.6f}  (L/D = {ROD_L/ROD_D})")

TIMESTEP = 0.005                   # lj
DUMP_EVERY = 5000
DT_DUMP = DUMP_EVERY * TIMESTEP   # 帧间隔 (τ)
S2_N_FRAMES = None
# fresnel 快照：间隔 stride、共 n 张 → snapshots/
SNAP_STRIDE = 50
SNAP_N = 10
SNAP_DIR = Path("snapshots")


## 1. 轨迹可视化

加压段 dump 为 `xsu ysu zsu`。**显示**时 `wrap`；后面算 $S_2$ 须保持 unwrap，**另开** Universe。


In [ ]:
u_view = load_lammps_universe(TOPO, TRAJ, dt_fs=DT_DUMP)
N_ROD = u_view.atoms.n_atoms // N_BEADS
print(f"atoms={u_view.atoms.n_atoms}, rods={N_ROD}, frames={u_view.trajectory.n_frames}")
u_view.trajectory.add_transformations(
    move_origin_to_corner,
    trans.wrap(u_view.atoms, compound="residues"),
)

view = nv.show_mdanalysis(u_view)
view.clear_representations()
view.add_spacefill(radiusType="vdw", radiusScale=0.35)
view.add_unitcell()
view


In [ ]:
# 导出某一帧为 PNG（默认最后一帧）
# save_nglview_frame(view, "last_frame.png")  # frame=-1 → last


## 2. 热力学输出

读全部 thermo 块。自定义列从 NVT 起才有 `time` / `volume` / `density`。横轴时间统一为 $\tau$。
**密度面板画相对密度 $\rho^*=\rho/\rho_{cp}$**（由 LAMMPS `density` 换算，见文首公式）。


In [ ]:
thermo_all = read_result_thermo("log.lammps", segment=None)
thermo = thermo_all.dropna(subset=["density", "time"]).copy()
thermo["rho_star"] = nem.relative_density(
    thermo["density"], kind="atom", n_beads=N_BEADS, L=ROD_L, D=ROD_D,
)
print("thermo blocks / rows:", thermo["segment"].nunique(), len(thermo))
print(thermo.groupby("segment").size())
print(f"rho* range: {thermo['rho_star'].min():.4f} → {thermo['rho_star'].max():.4f}")

seg_labels = {1: "NVT", 2: "NPT eq", 3: "compress"}

fig, axes = plt.subplots(2, 2, figsize=(9.0, 5.2), sharex=False)
panels = [
    ("temp", r"$T$ / $(\varepsilon/k_B)$"),
    ("press", r"$P$ / $(\varepsilon/\sigma^3)$"),
    ("rho_star", r"relative density $\rho^*$"),
    ("volume", r"$V$ / $\sigma^3$"),
]
for ax, (col, ylab) in zip(axes.ravel(), panels):
    for seg, g in thermo.groupby("segment"):
        ax.plot(g["time"], g[col], "-", label=seg_labels.get(int(seg), f"seg{seg}"), alpha=0.9)
    ax.set_xlabel(r"time / $\tau$")
    ax.set_ylabel(ylab)
    ax.legend(fontsize=8, loc="best")
fig.tight_layout()
fig.savefig("result_thermo.png", bbox_inches="tight")
print("saved: result_thermo.png")
plt.show()

compress = thermo[thermo["segment"] == thermo["segment"].max()].reset_index(drop=True)
print(
    "compress: "
    f"T={compress['temp'].mean():.3f}, "
    f"P {compress['press'].iloc[0]:.3f} → {compress['press'].iloc[-1]:.3f}, "
    f"ρ* {compress['rho_star'].iloc[0]:.3f} → {compress['rho_star'].iloc[-1]:.3f}"
)


## 3. 相对密度–时间 / 相对密度–压力（相变路径）

只画**生产（加压）段**，一行两列：左 $\rho^*(t)$，右 $\rho^*(P)$。
$\rho^*(t)$ 上常能看到 **nematic–smectic**、**smectic–crystal** 的两次密度跳跃；
**isotropic–nematic** 在密度上往往不明显，需靠下一节的 $S_2$。
右图瞬时压力涨落大，叠加滚动平均后，对照左图可粗估转变压力（教学定性，不强求精确相界）。


In [ ]:
# 生产段 = 最后一个自定义 thermo 块（加压 compress）
prod_pr = compress.copy()
SMOOTH_WIN = 51  # 可调；宜为奇数
min_periods = max(5, SMOOTH_WIN // 5)
p_s = prod_pr["press"].rolling(SMOOTH_WIN, center=True, min_periods=min_periods).mean()
rho_s = prod_pr["rho_star"].rolling(SMOOTH_WIN, center=True, min_periods=min_periods).mean()

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(9.0, 3.6))

ax0.plot(prod_pr["time"], prod_pr["rho_star"], "-", color="0.75", lw=0.8, alpha=0.55, label="raw")
ax0.plot(prod_pr["time"], rho_s, "-", color="C0", lw=2.0, label=f"rolling mean (n={SMOOTH_WIN})")
ax0.set_xlabel(r"time / $\tau$")
ax0.set_ylabel(r"relative density $\rho^*$")
ax0.legend(fontsize=8)

ax1.plot(prod_pr["press"], prod_pr["rho_star"], "-", color="0.75", lw=0.8, alpha=0.55, label="raw")
ax1.plot(p_s, rho_s, "-", color="C0", lw=2.0, label=f"rolling mean (n={SMOOTH_WIN})")
ax1.set_xlabel(r"pressure $P$ / $(\varepsilon/\sigma^3)$")
ax1.set_ylabel(r"relative density $\rho^*$")
ax1.legend(fontsize=8)

fig.tight_layout()
fig.savefig("result_density.png", bbox_inches="tight")
print("saved: result_density.png")
plt.show()


## 4. 向列序 $S_2$

每根棒用首尾珠算 $\hat{\mathbf u}$，构 $Q$ 张量，最大本征值即 $S_2$（见 `_nematic_order.py`）。
轨迹须 **unwrap**（本例 dump 已是 `xsu ysu zsu`）。

$S_2$ 能清楚标出 **isotropic–nematic**（密度跳跃不明显的那一段）；
随后 $\rho^*$ 上的跳跃则更多对应层状 / 晶体方向的有序化。横轴密度同样用相对密度 $\rho^*$。


In [ ]:
u_s2 = load_lammps_universe(TOPO, TRAJ, dt_fs=DT_DUMP)
N_ROD = u_s2.atoms.n_atoms // N_BEADS

s2_res = nem.compute_s2_trajectory(
    u_s2,
    n_beads=N_BEADS,
    dt=DT_DUMP,
    n_frames=S2_N_FRAMES,
    density_particles="rods",
    rod_L=ROD_L,
    rod_D=ROD_D,
)
s2_sum = nem.summarize_s2(s2_res)
print(f"frames={s2_res['n_frames']}, rods={s2_res['n_rods']}, rho_cp={s2_res['rho_cp']:.6f}")
print(
    f"S2 early→late: {s2_sum['S2_early']:.3f} → {s2_sum['S2_late']:.3f}; "
    f"ρ* early→late: {s2_sum['rho_star_early']:.4f} → {s2_sum['rho_star_late']:.4f}"
)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(9.0, 3.2))
ax0.plot(s2_res["times"], s2_res["S2"], "-")
ax0.set_xlabel(r"time / $\tau$")
ax0.set_ylabel(r"$S_2$")
ax0.set_title(r"$S_2(t)$ (compress)")

ax1.plot(s2_res["rho_star"], s2_res["S2"], "-")
ax1.set_xlabel(r"relative density $\rho^*$")
ax1.set_ylabel(r"$S_2$")
ax1.set_title(r"$S_2$ vs $\rho^*$")
fig.tight_layout()
fig.savefig("result_s2.png", bbox_inches="tight")
print("saved: result_s2.png")
plt.show()


## 5. 结果汇总


In [ ]:
summary = pd.DataFrame([
    {"quantity": "T_compress_mean", "value": compress["temp"].mean(), "unit": "ε/k_B"},
    {"quantity": "P_start", "value": compress["press"].iloc[0], "unit": "ε/σ³"},
    {"quantity": "P_end", "value": compress["press"].iloc[-1], "unit": "ε/σ³"},
    {"quantity": "rho_star_start", "value": compress["rho_star"].iloc[0], "unit": "-"},
    {"quantity": "rho_star_end", "value": compress["rho_star"].iloc[-1], "unit": "-"},
    {"quantity": "rho_cp", "value": RHO_CP, "unit": "1/σ³"},
    {"quantity": "S2_early", "value": s2_sum["S2_early"], "unit": "-"},
    {"quantity": "S2_late", "value": s2_sum["S2_late"], "unit": "-"},
    {"quantity": "S2_mean", "value": s2_sum["S2_mean"], "unit": "-"},
    {"quantity": "n_rods", "value": s2_res["n_rods"], "unit": "-"},
    {"quantity": "n_frames_S2", "value": s2_res["n_frames"], "unit": "-"},
])
summary.to_csv("summary_self_assembly.csv", index=False)
summary


## 6. Fresnel 渲染（快照序列）

对这类**粗粒化颗粒**体系，[Fresnel](https://fresnel.readthedocs.io/)（[GitHub](https://github.com/glotzerlab/fresnel) / [Glotzer lab](https://glotzerlab.engin.umich.edu/fresnel/)）是很合适的路径追踪渲染手段。安装见 [分子模拟工作平台搭建](../01-技术文档/T01-分子模拟工作平台搭建.md)。

本例每隔 **50** 帧渲染 **10** 张，写入当前目录下的 `snapshots/`（`snap_f0000.png`, `snap_f0050.png`, …）。渲染前按 **residue（整根棒）** wrap（`compound="residues"`），避免跨周期边界的棒被拆断；默认按 mol（棒）着色；下一单元可改按 $|\hat u\cdot\hat n|$。


In [ ]:
u_fr = load_lammps_universe(TOPO, TRAJ, dt_fs=DT_DUMP)
print(f"trajectory frames: {u_fr.trajectory.n_frames}")
print(f"render frames: {list(range(0, SNAP_STRIDE * SNAP_N, SNAP_STRIDE))}")

snap_paths = render_snapshot_series(
    u_fr,
    outdir=SNAP_DIR,
    n_frames=SNAP_N,
    stride=SNAP_STRIDE,
    start=0,
    color_method="mol",
    prefix="snap",
    view="ISO",
    radius=0.5,
    size=(600, 600),
    samples=(32, 16),
)
snap_paths


In [ ]:
# 可选：同一组帧按 |u·n| 着色（写入 snapshots/align_*.png）
align_paths = render_snapshot_series(
    u_fr,
    outdir=SNAP_DIR,
    n_frames=SNAP_N,
    stride=SNAP_STRIDE,
    start=0,
    color_method="atom_props",  # 每帧自动算 |u·n|
    n_beads=N_BEADS,
    prefix="align",
    view="ISO",
    radius=0.5,
    size=(600, 600),
    samples=(32, 16),
)
align_paths


In [ ]:
from IPython.display import Image, display
# 预览首 / 末两张（mol 着色）
display(Image(filename=str(snap_paths[0])))
display(Image(filename=str(snap_paths[-1])))
